# UCS420 Assignment 7 — NLP-I
**Name:** Aastha Tyagi  
**Roll No.:** 1024160131  
**Fingerprint** is printed after `generate_dataset()` (must match the starter).

`generate_dataset()` and `process_ticket()` are copied **unchanged** from `UCS420_LA9_Starter.ipynb`.


In [ ]:
ROLL_NO = 1024160131
NAME = "Aastha Tyagi"

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"
print("NLTK resources ready")


### Dataset generator — do not modify


In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint


### Preprocessing pipeline — do not modify


In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}


In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}")
print("Focus tickets:", FOCUS)
print()
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)


---
## Q1. Preprocessing
Run `process_ticket()` on every ticket. Table columns: sentences, tokens, punctuation tokens, stopwords, unique tokens, final tokens, plus a TOTAL row. Then show the actual lists for one selected ticket.


In [ ]:
from IPython.display import display

results = []
processed_tickets = {}
for _, row in df.iterrows():
    processed = process_ticket(row["Ticket_Text"])
    processed_tickets[row["Ticket_ID"]] = processed
    results.append({
        "Ticket_ID": row["Ticket_ID"],
        "Sentences": len(processed["sentences"]),
        "Tokens": len(processed["tokens"]),
        "Punctuation Tokens": len(processed["punct"]),
        "Stopwords": len(processed["stops"]),
        "Unique Tokens": len(set(processed["tokens"])),
        "Final Tokens": len(processed["final"]),
    })

summary_df = pd.DataFrame(results)
totals = {"Ticket_ID": "TOTAL"}
for col in ["Sentences", "Tokens", "Punctuation Tokens", "Stopwords", "Unique Tokens", "Final Tokens"]:
    totals[col] = int(summary_df[col].sum())
summary_df = pd.concat([summary_df, pd.DataFrame([totals])], ignore_index=True)
display(summary_df)

selected_id = FOCUS[0] if FOCUS else df.iloc[0]["Ticket_ID"]
selected = processed_tickets[selected_id]
print("\nSelected ticket:", selected_id)
print("Original:", df.loc[df["Ticket_ID"] == selected_id, "Ticket_Text"].iloc[0])
print("Sentences:", selected["sentences"])
print("Tokens:", selected["tokens"])
print("Punctuation tokens:", selected["punct"])
print("Stopwords:", selected["stops"])
print("Unique tokens:", sorted(set(selected["tokens"])))
print("Final tokens:", selected["final"])


---
## Q2. Regex extraction and replacement
Patterns are checked on the generated tickets **and** on `TEST_LINES`.


In [ ]:
import re

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

LONG_WORD = r"\b[A-Za-z]{6,}\b"
# versions/decimals first so 2.0.1 and 95.5 stay whole
NUMBER = r"\b\d+(?:\.\d+)+\b|\b\d+\b"
CAPITAL = r"\b[A-Z][a-zA-Z]*\b"
VOWEL = r"\b[AEIOUaeiou][A-Za-z]*\b"
EMAIL = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
URL = r"(?:https?://|www\.)[^\s<>]+"
PHONE = r"(?:\+91[-\s]?)?\d{5}[-\s]?\d{5}|\d{4}-\d{3}-\d{4}"

def show_findalls(label, pattern, texts):
    print(f"\n===== {label} =====")
    for i, text in enumerate(texts, 1):
        print(f"[{i}] {re.findall(pattern, text)}")

def redact(text):
    text = re.sub(EMAIL, "<EMAIL>", text)
    text = re.sub(URL, "<URL>", text)
    text = re.sub(PHONE, "<PHONE>", text)
    return text

corpus = df["Ticket_Text"].tolist() + TEST_LINES
show_findalls("words longer than 5 letters", LONG_WORD, corpus)
show_findalls("numbers (decimals/versions kept whole)", NUMBER, corpus)
show_findalls("capitalized words", CAPITAL, corpus)
show_findalls("words beginning with a vowel", VOWEL, corpus)

print("\n===== redacted =====")
for i, text in enumerate(corpus, 1):
    print(f"[{i}] {redact(text)}")

print("\nSanity check on TEST_LINES redaction:")
for line in TEST_LINES:
    print(redact(line))


---
## Q3. Custom tokenization
One `RegexpTokenizer` pattern. Order matters: URLs, emails and phones are matched before ordinary words.


In [ ]:
from nltk.tokenize import RegexpTokenizer, word_tokenize

TOKEN_PATTERN = (
    r"(?:https?://|www\.)[^\s<>]+|"
    r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|"
    r"(?:\+91[-\s]?)?\d{5}[-\s]?\d{5}|\d{4}-\d{3}-\d{4}|"
    r"\d+(?:\.\d+)+|"
    r"[A-Za-z]+(?:[-'][A-Za-z]+)*|"
    r"\d+|"
    r"[^\w\s]"
)
_tokenizer = RegexpTokenizer(TOKEN_PATTERN)

def my_tokenize(text):
    return _tokenizer.tokenize(text)

print("TEST_LINES")
for i, line in enumerate(TEST_LINES, 1):
    print(f"\nLINE {i}: {line}")
    print(" my_tokenize :", my_tokenize(line))
    print(" word_tokenize:", word_tokenize(line))

rows = []
for _, row in df.iterrows():
    a, b = my_tokenize(row["Ticket_Text"]), word_tokenize(row["Ticket_Text"])
    rows.append({
        "Ticket_ID": row["Ticket_ID"],
        "different": a != b,
        "my_tokenize": a,
        "word_tokenize": b,
    })
cmp_df = pd.DataFrame(rows)
display(cmp_df[["Ticket_ID", "different", "my_tokenize", "word_tokenize"]])
print("Tickets tokenized differently:", int(cmp_df["different"].sum()), "/", len(cmp_df))


**Three differences (this dataset):**
1. Contractions: `word_tokenize` splits `couldn't` / `It's` into `could` + `n't` and `It` + `'s`. `my_tokenize` keeps `couldn't` and `It's` as one token.
2. Hyphens: `e-mail` stays one token in `my_tokenize`; NLTK often splits on `-`.
3. Technical numbers: `2.0.1` is one token in `my_tokenize` because the pattern matches dotted versions; `word_tokenize` may split around `.`.


---
## Q4. Stemming and lemmatization
Compared on **unique final tokens**. The Regexp stemmer uses suffix `-ing` because this corpus has several process verbs (`working`, `disconnecting`, `uploading`, `opening`, `restarting`).


In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

unique_final = sorted(set(t for p in processed_tickets.values() for t in p["final"]))
print("Unique final tokens:", len(unique_final))
print(unique_final)

porter = PorterStemmer()
lancaster = LancasterStemmer()
# suffix chosen from the dataset: -ing appears on working/disconnecting/uploading/...
regexp = RegexpStemmer("ing$|ed$|s$", min=4)
wnl = WordNetLemmatizer()

def wn_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

tagged = pos_tag(unique_final)
lemma_map = {w: wnl.lemmatize(w, wn_pos(tag)) for w, tag in tagged}

rows = []
for w in unique_final:
    p, l, r, lem = porter.stem(w), lancaster.stem(w), regexp.stem(w), lemma_map[w]
    if len({p, l, r, lem, w}) > 1:
        rows.append({
            "word": w, "POS": dict(tagged)[w],
            "Porter": p, "Lancaster": l, "Regexp": r, "WordNet+POS": lem,
        })
diff_df = pd.DataFrame(rows)
display(diff_df)
print("Words where at least one method differs:", len(diff_df))


---
## Q5. Corpus statistics
S1 = all tokens · S2 = tokens without punctuation · S3 = final tokens · S4 = S3 after Porter stemming.
TTR = unique / total.


In [ ]:
from collections import Counter
from nltk import bigrams
import math
import matplotlib.pyplot as plt

S1 = [t for p in processed_tickets.values() for t in p["tokens"]]
S2 = [t for t in S1 if not is_punct(t)]
S3 = [t for p in processed_tickets.values() for t in p["final"]]
S4 = [porter.stem(t) for t in S3]

def stats(label, seq):
    total, uniq = len(seq), len(set(seq))
    ttr = uniq / total if total else 0
    print(f"{label:4}  total={total:4d}  unique={uniq:4d}  TTR={ttr:.4f}")
    return total, uniq, ttr

print("Corpus statistics")
for lab, seq in [("S1", S1), ("S2", S2), ("S3", S3), ("S4", S4)]:
    stats(lab, seq)

print("\nTop 10 S1:", Counter(S1).most_common(10))
print("Top 10 S3:", Counter(S3).most_common(10))
print("Top 5 bigrams S1:", Counter(bigrams(S1)).most_common(5))
print("Top 5 bigrams S3:", Counter(bigrams(S3)).most_common(5))

freq = Counter(S3)
ranked = sorted(freq.values(), reverse=True)
ranks = list(range(1, len(ranked) + 1))
plt.figure(figsize=(6, 4))
plt.plot([math.log(r) for r in ranks], [math.log(f) for f in ranked], marker="o")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf plot — S3 final tokens")
plt.tight_layout()
plt.show()


---
## Q6. Interpretation

**(a)** TTR changes because each set throws away a different kind of repetition. S1 is dominated by repeated stopwords (`the`, `i`) and punctuation (`.`), so unique/total is smaller. Dropping punctuation (S2) and then stopwords (S3) shrinks the denominator faster than the numerator, so TTR usually **rises**. Porter stemming on S3 maps variants such as `working`/`worked`/`works` onto one stem, so unique count (and TTR) **falls** from S3 to S4.

**(b)** On T11, `couldn't` is one token for `my_tokenize` but `word_tokenize` yields `could` and `n't`. NLTK's Treebank-style tokenizer splits clitics; the custom regex treats `'` as part of a word.

**(c)** Lancaster stemming is too aggressive on short IT words. Example: `server` → `serv`, which is no longer the same concept (a machine vs. the verb *serve*).

**(d)** POS-aware lemmatization of `was` / `is` → `be`, while Porter gives `wa` / `is`. The lemma keeps the verb identity; the stem does not.
